# Week 8: Using Transformer Models


## Getting started
If working on your own machine, make sure the huggingface transformers package is installed

`conda install -c huggingface transformers`

or

`pip install transformers`

Of course, if working on Google Colab, you won't need to do this.  Whatever environment you are using check whether the following code runs.  It should output a negative label with a high score!


In [63]:
from transformers import pipeline
print(pipeline('sentiment-analysis')('I hate you'))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/resolve/714eb0f/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/714eb0fa89d2f80546fda750413ed43d93601a13/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8359.51it/s]
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/714eb0f/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetune

[{'label': 'NEGATIVE', 'score': 0.9991129040718079}]


This `pipeline` is a multi-step function:
1. **Model Download and Loading:** A pre-trained model can be set but defaults to DistilBERT fine-tuned on the SST-2 dataset.
2. **Tokenization:** Breaks the input sentence down into tokens and add any special tokens such as `[CLS]` or `[SEP]`, maps tokens to vocab IDs and repackages the data into a tensor reading for a NN. 
3. **The Forward Pass:** The tensor is then passed through a transformer. This produces the embeddings for each token (and the `[CLS]`) and from which is passes through the self-attention mechanism calculting how much every word relates to every other word.
4. **The Classification Head:** The pipeline passes the `[CLS]` token into the final classification linear layer which outputs raw logits. 
5. **Post-Processing:** Finally it passes the logits to the softmax which squashes the raw scores into probabilities that add up to 100%.

The following is adapted from an older version of the huggingface quickstart to transformers tutorial 
https://huggingface.co/transformers/v2.4.0/quickstart.html
We will be looking at the BERT introduction (but feel free to have a look at GPT2 etc as well!)

First of all we need some key imports.  We are going to be using the pre-trained bert-base-uncased model so this cell instantiates a tokenizer for this model.  Logging is also switched on so we can see more of what's going on. The first time you run it, the model will be downloaded and cached.  The cached version will be used on subsequent runs, if it is available (not on Google CoLab).

In [64]:
import torch
from transformers import BertTokenizer, BertModel, BertForMaskedLM

# OPTIONAL: if you want to have more information on what's happening under the hood, activate the logger as follows
import logging
logging.basicConfig(level=logging.INFO)

# Load pre-trained model tokenizer (vocabulary)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


In [65]:
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

Now we are going to tokenize some text.  This will demonstrate the 'wordpiece' vocabulary used by BERT as well as the fact that we need to introduce special `[CLS]` and `[SEP]` tokens in the input.

In [66]:
# Tokenize input
text = "[CLS] Who was elected as British prime minister in 1951? [SEP] Sir Winston Leonard Spencer Churchill was a British politician, statesman, army officer and writer, who was Prime Minister of the United Kingdom from 1940 to 1945 and again from 1951 to 1955. [SEP]"
tokenized_text = tokenizer.tokenize(text)
print(tokenized_text)

['[CLS]', 'who', 'was', 'elected', 'as', 'british', 'prime', 'minister', 'in', '1951', '?', '[SEP]', 'sir', 'winston', 'leonard', 'spencer', 'churchill', 'was', 'a', 'british', 'politician', ',', 'statesman', ',', 'army', 'officer', 'and', 'writer', ',', 'who', 'was', 'prime', 'minister', 'of', 'the', 'united', 'kingdom', 'from', '1940', 'to', '1945', 'and', 'again', 'from', '1951', 'to', '1955', '.', '[SEP]']


In [67]:
# Tokenize input
text = "[CLS] What are igneous rocks? [SEP] Igneous rocks form when hot , molten rock crystallizes and solidifies. [SEP] "
tokenized_text= tokenizer.tokenize(text)
print(tokenized_text)

['[CLS]', 'what', 'are', 'ign', '##eous', 'rocks', '?', '[SEP]', 'ign', '##eous', 'rocks', 'form', 'when', 'hot', ',', 'molten', 'rock', 'crystal', '##li', '##zes', 'and', 'solid', '##ifies', '.', '[SEP]']


Note that the tokenizer is not breaking down all words according to their morphology -- only rare words.  Reasonably frequent words such as `elected` are left as whole words.  Rarer words such as `solidifies` are broken down.

Now we are going to mask out one of the words in the text.  For the purposes of this demonstration, I have chosen token 11 but you could try different tokens.  Remember that during training the tokens to mask are chosen randomly.


In [68]:
# Mask a token that we will try to predict back with `BertForMaskedLM`
masked_index = 10
print(tokenized_text[10])
tokenized_text[masked_index] = '[MASK]'
print(tokenized_text)

rocks
['[CLS]', 'what', 'are', 'ign', '##eous', 'rocks', '?', '[SEP]', 'ign', '##eous', '[MASK]', 'form', 'when', 'hot', ',', 'molten', 'rock', 'crystal', '##li', '##zes', 'and', 'solid', '##ifies', '.', '[SEP]']


In [69]:
print(len(tokenized_text))

25


We are now going to try to use the masked language model to predict this word.

First we need to convert the input into a list of word index ids.

In [70]:
# Convert token to vocabulary indices
indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
print(indexed_tokens)

[101, 2054, 2024, 16270, 14769, 5749, 1029, 102, 16270, 14769, 103, 2433, 2043, 2980, 1010, 23548, 2600, 6121, 3669, 11254, 1998, 5024, 14144, 1012, 102]


We need segment ids to define whether a token is in the first or second sentence.

In [71]:
def make_segment_ids(list_of_tokens):
    #this function assumes that up to and including the first '[SEP]' is the first segment, anything afterwards is the second segment
    current_id=0
    segment_ids=[]
    for token in list_of_tokens:
        segment_ids.append(current_id)
        if token == '[SEP]':
            current_id+=1
    return segment_ids

segment_ids=make_segment_ids(tokenized_text)
print(segment_ids)

[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [72]:
# Convert inputs to PyTorch tensors
#this just wraps things up in multi-dimensional tensors rather than as flat lists.
tokens_tensor = torch.tensor([indexed_tokens])
segments_tensors = torch.tensor([segment_ids])
print(tokens_tensor)
print(segments_tensors)

tensor([[  101,  2054,  2024, 16270, 14769,  5749,  1029,   102, 16270, 14769,
           103,  2433,  2043,  2980,  1010, 23548,  2600,  6121,  3669, 11254,
          1998,  5024, 14144,  1012,   102]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]])


Now we need to encode the input using the bert-base-uncased model


In [73]:
# Check if CUDA (NVIDIA GPU) is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Check for NVIDIA GPU (cuda) or Apple Silicon GPU (mps)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cpu
Using device: mps


This specific block of code is the "Inference Step." This is the moment where you actually pass your processed data through the neural network to get the mathematical "thoughts" (vectors) that BERT has about your sentence.
- BERT produces several different types of results. The lab focuses on `outputs[0]`, often called the `last_hidden_state`.
- `(batch size, sequence length, model hidden dimension)`
    * **Batch size (1):** You only sent 1 set of sentences.
    * **Sequence length:** If your input has 60 tokens, there are 60 positions here.
    * **Hidden dimension (768):** This is the most important part. BERT represents every single word in your sentence as a list of 768 numbers.

These 768 numbers are the Contextualized Embeddings. They are high-dimensional coordinates that capture the exact meaning of that word in that specific Churchill-related context.

In [74]:
# Load pre-trained model (weights)
model = BertModel.from_pretrained('bert-base-uncased')

# Set the model in evaluation mode to deactivate the DropOut modules
# This is IMPORTANT to have reproducible results during evaluation!
model.eval()

# If you have a GPU, put everything on cuda - otherwise comment this out to run on CPU
# Check for NVIDIA GPU (cuda) or Apple Silicon GPU (mps)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Move your model and tensors just like before
model.to(device)
tokens_tensor = tokens_tensor.to(device)
segments_tensors = segments_tensors.to(device)

# Predict hidden states features for each layer
with torch.no_grad():
    # See the models docstrings for the detail of the inputs
    outputs = model(tokens_tensor, token_type_ids=segments_tensors)
    # Transformers models always output tuples.
    # See the models docstrings for the detail of all the outputs
    # In our case, the first element of outputs is the output of the last layer of the Bert model (all tokens)
    # the second element of outputs, outputs[1] is actually just a "pooled_output" representation of the CLS token (rather than all tokens) - however this involves an extra layer which is why it is not the same as the first element in outputs[0]!
    encoded_layers = outputs[0]
# We have encoded our input sequence in a FloatTensor of shape (batch size, sequence length, model hidden dimension)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13235.86it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

Using device: mps


In [75]:
# We have encoded our input sequence in a FloatTensor of shape (batch size, sequence length, model hidden dimension)
print(encoded_layers.shape)

torch.Size([1, 25, 768])


In [76]:
#outputs[1] is a representation of the CLS token of shape (batch size, model hidden dimension)
outputs[1].shape

torch.Size([1, 768])

In [77]:
outputs[1]

tensor([[-0.9909, -0.8393, -0.9960,  0.9870,  0.9608, -0.7787,  0.9874,  0.7262,
         -0.9908, -1.0000, -0.9321,  0.9985,  0.9956,  0.8476,  0.9892, -0.9731,
         -0.9388, -0.8946,  0.7198, -0.9230,  0.9475,  1.0000, -0.6464,  0.7893,
          0.8770,  0.9999, -0.9734,  0.9872,  0.9905,  0.8786, -0.9629,  0.7455,
         -0.9983, -0.6659, -0.9947, -0.9993,  0.8991, -0.9418, -0.6326, -0.6636,
         -0.9731,  0.8429,  1.0000,  0.2345,  0.8452, -0.7612, -1.0000,  0.7345,
         -0.9713,  0.9981,  0.9901,  0.9930,  0.8092,  0.9193,  0.8983, -0.8813,
          0.5981,  0.6350, -0.7381, -0.9264, -0.8636,  0.8174, -0.9874, -0.9727,
          0.9969,  0.9793, -0.7530, -0.7952, -0.6978,  0.4207,  0.9878,  0.7282,
         -0.6149, -0.9444,  0.9723,  0.7580, -0.7926,  1.0000, -0.9546, -0.9968,
          0.9727,  0.9792,  0.7685, -0.8853,  0.9199, -1.0000,  0.9217, -0.5914,
         -0.9976,  0.7747,  0.8924, -0.7403,  0.9005,  0.8271, -0.7948, -0.8763,
         -0.8411, -0.9910, -

We can predict the masked token as follows.  Using the last layer of the BERT model, we find the token id which maximises the prediction for the masked token.

In [78]:
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
model.eval()

# 1. Define the device dynamically
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

# 2. Move the model to the device
model.to(device)
model.eval()

# 3. CRITICAL: Move your input tensors to the SAME device
tokens_tensor = tokens_tensor.to(device)
segments_tensors = segments_tensors.to(device)

# Predict all tokens
with torch.no_grad():
    outputs = model(tokens_tensor, token_type_ids=segments_tensors)
    predictions = outputs[0]

        
# find the token id which maximises the prediction for the masked token and then convert this back to a word
predicted_index = torch.argmax(predictions[0, masked_index]).item()
predicted_token = tokenizer.convert_ids_to_tokens([predicted_index])[0]
print(predicted_token)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 14268.50it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


rocks


Did BERT correctly predict the masked token?

### Exercise 0
Mask each token in turn and see what BERT predicts.   How accurate are its predictions?  As an extension, you could look at masking multiple words in the sequence.

In [79]:
# Initializing the "Cloze" Expert
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
model.eval()

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 14864.81it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

`BertForMaskedLM`: Unlike the `BertModel` you used earlier (which just gives vectors), this version has a "Language Modeling" head on top. This head is specifically designed to output probabilities for every word in the 30,000-word vocabulary.

For the `BERTModel`, `output[0]` was the CLS token but for this MASKED model this index is `[Batch Size, Sequence Length, Vocabulary Size]` which in this example is `[1, 23, 30522]`
- `1:` one sentence
- `23:` 23 tokens (including [CLS], [SEP], and the words).
- `30522:` This is the most important part. For every one of those 23 tokens, BERT provides a score for all 30,522 words in its dictionary.

If `outputs[0]` were just the `[CLS]` token, the shape would only be `[1, 30522]`.

In [80]:
text = "[CLS] What are igneous rocks? [SEP] Igneous rocks form when hot , molten rock crystallizes and solidifies. [SEP] "


correct=0
n=0
for i in range(len(tokenized_text)):
    tokenized_text= tokenizer.tokenize(text)
    masked_index=i
    gold=tokenized_text[masked_index]
    tokenized_text[masked_index]='[MASK]'
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    segment_ids=make_segment_ids(tokenized_text)
    tokens_tensor = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segment_ids])

    # Predict all tokens
    with torch.no_grad():
        outputs = model(tokens_tensor, token_type_ids=segments_tensors)
        predictions = outputs[0]

        
    # find the token id which maximises the prediction for the masked token and then convert this back to a word
    predicted_index = torch.argmax(predictions[0, masked_index]).item() # grabs [sentence, the index of the masked word], from the 30k vocab argmax the biggest
    #torch.argmax return index (id) not the logit itself
    predicted_token = tokenizer.convert_ids_to_tokens([predicted_index])[0]
    print(f'Gold:"{gold}", Predicted: "{predicted_token}"')
    n+=1
    if predicted_token==gold:
        correct+=1
print("Accuracy: {}".format(str(correct/n)))

Gold:"[CLS]", Predicted: "."
Gold:"what", Predicted: "what"
Gold:"are", Predicted: "about"
Gold:"ign", Predicted: "ign"
Gold:"##eous", Predicted: "##eous"
Gold:"rocks", Predicted: "rocks"
Gold:"?", Predicted: "?"
Gold:"[SEP]", Predicted: """
Gold:"ign", Predicted: "ign"
Gold:"##eous", Predicted: "##eous"
Gold:"rocks", Predicted: "rocks"
Gold:"form", Predicted: "form"
Gold:"when", Predicted: "when"
Gold:"hot", Predicted: "solid"
Gold:",", Predicted: ","
Gold:"molten", Predicted: "hard"
Gold:"rock", Predicted: "rock"
Gold:"crystal", Predicted: "crystal"
Gold:"##li", Predicted: "##li"
Gold:"##zes", Predicted: "##zes"
Gold:"and", Predicted: "and"
Gold:"solid", Predicted: "solid"
Gold:"##ifies", Predicted: "##ifies"
Gold:".", Predicted: "."
Gold:"[SEP]", Predicted: """
Accuracy: 0.76


In [81]:
torch.argmax(predictions[0, masked_index])

tensor(1000)

In [82]:
predictions[0, masked_index]

tensor([-6.8663, -6.9542, -6.7119,  ..., -6.0662, -6.0948, -6.2417])

## Representing Sentential Meaning
We are going to be looking at different strategies for representing sentential meaning
* CLS token representation
* centroid/sum of output embeddings

The file `examples.txt` contains some example sentences.

### Exercise 1
Read in the sentences and store them as a list of sentences.  Add `[CLS]` and `[SEP]` tokens to the beginning and end of each and then pass them through the bert-base-uncased tokenizer

In [84]:
filename='examples.txt'
sentences=[]
with open(filename, 'r') as instream:
    for line in instream:
        sentences.append(line.rstrip())
#print(sentences)

inputsents=['[CLS] '+sent+' [SEP]' for sent in sentences]

tokenized_sents=[tokenizer.tokenize(sent) for sent in inputsents]
print(tokenized_sents)

[['[CLS]', 'the', 'boy', 'kicks', 'the', 'ball', '.', '[SEP]'], ['[CLS]', 'the', 'ball', 'kicks', 'the', 'boy', '.', '[SEP]'], ['[CLS]', 'the', 'child', 'kicks', 'the', 'ball', '.', '[SEP]'], ['[CLS]', 'the', 'ball', 'is', 'kicked', 'by', 'the', 'boy', '.', '[SEP]'], ['[CLS]', 'the', 'ball', 'is', 'kicked', '.', '[SEP]'], ['[CLS]', 'the', 'boy', 'kicks', '.', '[SEP]'], ['[CLS]', 'the', 'child', 'kicks', '.', '[SEP]'], ['[CLS]', 'the', 'boy', 'kicks', 'a', 'round', 'object', '.', '[SEP]'], ['[CLS]', 'the', 'male', 'child', 'kicks', 'the', 'ball', '.', '[SEP]'], ['[CLS]', 'the', 'boy', 'is', 'playing', 'football', '.', '[SEP]'], ['[CLS]', 'the', 'boy', 'hits', 'the', 'ball', '.', '[SEP]'], ['[CLS]', 'the', 'ball', 'hits', 'the', 'boy', '.', '[SEP]'], ['[CLS]', 'the', 'boy', 'is', 'hit', 'by', 'the', 'ball', '.', '[SEP]'], ['[CLS]', 'the', 'ball', 'is', 'hit', 'by', 'the', 'boy', '.', '[SEP]'], ['[CLS]', 'the', 'female', 'child', 'kicks', 'the', 'ball', '.', '[SEP]'], ['[CLS]', 'the', 'gi

In [85]:
for i,sent in enumerate(inputsents):
    print(i,sent)

0 [CLS] The boy kicks the ball. [SEP]
1 [CLS] The ball kicks the boy. [SEP]
2 [CLS] The child kicks the ball. [SEP]
3 [CLS] The ball is kicked by the boy. [SEP]
4 [CLS] The ball is kicked. [SEP]
5 [CLS] The boy kicks. [SEP]
6 [CLS] The child kicks. [SEP]
7 [CLS] The boy kicks a round object. [SEP]
8 [CLS] The male child kicks the ball. [SEP]
9 [CLS] The boy is playing football. [SEP]
10 [CLS] The boy hits the ball. [SEP]
11 [CLS] The ball hits the boy. [SEP]
12 [CLS] The boy is hit by the ball. [SEP]
13 [CLS] The ball is hit by the boy. [SEP]
14 [CLS] The female child kicks the ball. [SEP]
15 [CLS] The girl kicks the ball. [SEP]
16 [CLS] The child plays with dolls. [SEP]
17 [CLS] The female child plays with dolls. [SEP]
18 [CLS] The male child plays with dolls. [SEP]
19 [CLS] The girl plays with dolls. [SEP]
20 [CLS] The boy plays with dolls. [SEP]
21 [CLS] The boy is kicking the ball. [SEP]
22 [CLS] The boy is not kicking the ball. [SEP]
23 [CLS] All boys kick balls. [SEP]
24 [CLS] Ev

---

When encoding sentences, it is actually more typical to pool the hidden states for each layer (at depth n) rather than the output layer.  We can access the hidden states of the model using `output_hidden_states=True` 

- The Lower Layers (near the input) tend to capture surface-level features (like grammar and part-of-speech).
- The Middle Layers capture syntactic information (sentence structure).
- The Final Layers (near the output) capture the most complex semantic information, but they are also very biased toward the specific task the model was trained on (like predicting a masked word).

By setting `output_hidden_states=True`, your `outputs` tuple becomes much larger. Instead of just the last layer, you get a tuple of 13 tensors:

In [86]:
print(model.config.num_hidden_layers)

12


In [87]:
model = BertModel.from_pretrained('bert-base-uncased')

model.eval()

# Predict hidden states features for each layer
with torch.no_grad():
    # See the models docstrings for the detail of the inputs
    outputs = model(tokens_tensor, token_type_ids=segments_tensors,output_hidden_states=True)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13127.40it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

Convert to tuple from `BaseModelOutputWithPooling` object. The object is actually much better to work with and provides quick access to information: last_layer = `outputs.last_hidden_state` , `all_layers = outputs.hidden_states`. However, many older tutorials and labels just access the info through indexing which is version agnostic. 

The tuple stacks the information as:
- **Index 0:** Last Hidden State (the vectors for every word).
- **Index 1:** Pooler Output (the summary of the `[CLS]` token).
- **Index 2:** Hidden States (only if you set `output_hidden_states=True`).
- **Index 3:** Attentions (only if you set `output_attentions=True`)

Could easily be unpacked: `last_hidden, pooled, all_hidden = outputs.to_tuple()`

Note that as there are 12 layers `all_hidden` will be a nested tuple:
- Index [0]: The Embedding Layer (Layer 0). This is the "raw" word representation before it has gone through any Transformer blocks.
- Index [1]: The output of the 1st Layer.
- Index [12]: The output of the 12th Layer (the final output).

Every one of those 13 indices contains its own 3D tensor. If you have 1 sentence with 10 tokens, every single layer index will contain a tensor of shape: `[Batch: 1, Sequence: 10, Hidden: 768]`

#### Typical Pooling Strats
- **Second-to-last layer (index [-2]):** Often avoids "over-specialization" that happens in the very last layer
- **Concatenation:** Taking the vectors from the last 4 layers and gluing them together to make a massive vector of $4 \times 768 = 3072$ numbers.
- **Summing/Averaging:** Adding the vectors from the last 4 layers together to create a single "super-vector" of 768 numbers.

In [88]:
outputs.to_tuple()

(tensor([[[-0.6770,  0.9460, -0.5043,  ..., -0.5411,  0.2880,  0.6554],
          [ 0.3894, -0.1123,  0.0627,  ...,  0.2348, -0.2624, -0.7900],
          [ 0.3382,  0.4643,  0.5102,  ..., -0.3201,  0.0523,  0.6370],
          ...,
          [ 0.3637,  0.2680,  0.4165,  ..., -0.1186, -0.3448,  0.1381],
          [ 0.6523, -0.0097, -0.2530,  ...,  0.0553, -0.6688, -0.4759],
          [ 0.1958,  0.3457,  0.2455,  ..., -0.0109, -0.0633,  0.1034]]]),
 tensor([[-0.9894, -0.8646, -0.9975,  0.9868,  0.9707, -0.8055,  0.9864,  0.7454,
          -0.9937, -1.0000, -0.9569,  0.9987,  0.9958,  0.8907,  0.9904, -0.9714,
          -0.9318, -0.9199,  0.7476, -0.8990,  0.9496,  1.0000, -0.6833,  0.8039,
           0.8966,  0.9999, -0.9735,  0.9875,  0.9899,  0.8809, -0.9644,  0.7739,
          -0.9986, -0.6912, -0.9968, -0.9994,  0.9149, -0.9360, -0.6885, -0.6479,
          -0.9761,  0.8580,  1.0000,  0.3837,  0.8634, -0.7817, -1.0000,  0.7828,
          -0.9685,  0.9987,  0.9940,  0.9951,  0.8279,  0.

In [89]:
len(outputs)

3

In [90]:
print(len(outputs))
for i in range(len(outputs)):
    try:
        print(outputs[i].shape)
    except:
        print(len(outputs[i]))

3
torch.Size([1, 25, 768])
torch.Size([1, 768])
13


Here:
* outputs[0] contains the output representation of each token
* outputs[1] is representation of the first token (after being put through an additional layer)
* outputs[2] is a a tuple.  Each element is the hidden layer at depth n.  If we want the last layer then we need outputs[2][-1]


In [91]:
#outputs[2][-1] is the last hidden layer also output as outputs[0]
outputs[2][-1]

tensor([[[-0.6770,  0.9460, -0.5043,  ..., -0.5411,  0.2880,  0.6554],
         [ 0.3894, -0.1123,  0.0627,  ...,  0.2348, -0.2624, -0.7900],
         [ 0.3382,  0.4643,  0.5102,  ..., -0.3201,  0.0523,  0.6370],
         ...,
         [ 0.3637,  0.2680,  0.4165,  ..., -0.1186, -0.3448,  0.1381],
         [ 0.6523, -0.0097, -0.2530,  ...,  0.0553, -0.6688, -0.4759],
         [ 0.1958,  0.3457,  0.2455,  ..., -0.0109, -0.0633,  0.1034]]])

In [92]:
outputs[0]

tensor([[[-0.6770,  0.9460, -0.5043,  ..., -0.5411,  0.2880,  0.6554],
         [ 0.3894, -0.1123,  0.0627,  ...,  0.2348, -0.2624, -0.7900],
         [ 0.3382,  0.4643,  0.5102,  ..., -0.3201,  0.0523,  0.6370],
         ...,
         [ 0.3637,  0.2680,  0.4165,  ..., -0.1186, -0.3448,  0.1381],
         [ 0.6523, -0.0097, -0.2530,  ...,  0.0553, -0.6688, -0.4759],
         [ 0.1958,  0.3457,  0.2455,  ..., -0.0109, -0.0633,  0.1034]]])

In [25]:
#so if you want the penultimate hidden layer you need outputs[2][-2]
outputs[2][-2]

tensor([[[-0.3338,  0.5738, -0.5777,  ..., -0.1215, -0.3504,  1.1074],
         [ 0.4044, -0.2317, -0.2335,  ...,  0.4218, -0.3578, -1.5020],
         [-0.2562,  0.3409, -0.2989,  ..., -0.5693,  0.0474,  1.3045],
         ...,
         [ 0.8100, -0.0503,  0.3418,  ..., -0.1121, -0.2673,  0.5875],
         [ 0.0457,  0.0066, -0.0341,  ...,  0.0187, -0.0104, -0.0071],
         [ 0.4002,  0.4515,  0.5819,  ...,  0.0999, -0.6354,  0.0041]]])

### Exercise 2

Encode each sentence using the output representation for its CLS token - note that you do not need to mask the CLS token.  We are just interested in the output layer embedding for this token.  

You can use `outputs[0][0]` or `outputs[1]` as a representation of the CLS token - but you will get different results as outputs[1] has gone through an additional layer (trained for next sentence prediction during fine-tuning and classification IF the model has been fine-tuned).
- `outputs[2]` is a tuple of 13 tensors (Layer 0 through Layer 12).
- `outputs[2][12]` is exactly the same data as `outputs[0]` as you are manually selecting a hidden layer
- Furthermore `outputs[0][0]` is grabbing the first sentence of the final hidden layer
- `outputs[0][0][0]` would select the first token, i.e. the `[CLS]` token
- `outputs[1]` is known as the pooled output and has need specifically trainined using Next Sentence Prediction (NSP). Many researchers find this representatoin too specalised for the specific tasks BERT was trainined on. 

Use cosine similarity to determine all pairs similarities for the **sentences** based on their [CLS] token vector. 

Identify the 10 most similar pairs of sentences using this sentence encoding

In [93]:
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

encoded=[]
for sent in tokenized_sents:
    indexed_tokens = tokenizer.convert_tokens_to_ids(sent)
    segment_ids=make_segment_ids(sent)
    tokens_tensor = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segment_ids])

    # Predict all tokens
    with torch.no_grad():
        outputs = model(tokens_tensor, token_type_ids=segments_tensors)
        predictions = outputs[0]  #0th output is the last hidden layer
        
    cls=predictions[0,0]  #0th batch (sentence) and 0th token (cls), 
    encoded.append(cls)

print(len(encoded))

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12522.75it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

35


In [108]:
## this is a handy way of finding the cosine similarity between two tensors
# see https://pytorch.org/docs/stable/generated/torch.nn.CosineSimilarity.html
cos = torch.nn.CosineSimilarity(dim=0, eps=1e-6)

#you use this as:
# print(encoded_layers[0,0],encoded_layers[0,1])
output=cos(encoded_layers[0,0],encoded_layers[0,1])
print(output.item())

0.13376513123512268


In [118]:
sims=[]
for posA in encoded:
    this_sims=[]
    for posB in encoded:
        
        output=cos(posA,posB)
        this_sims.append(output.item())
    sims.append(this_sims)
    
print(sims)
print(len(sims))

[[1.000000238418579, 0.9455858469009399, 0.9646621942520142, 0.8514186143875122, 0.82864910364151, 0.940155029296875, 0.9227979779243469, 0.9353492856025696, 0.9003759026527405, 0.8708140850067139, 0.9646227955818176, 0.9289813041687012, 0.8759910464286804, 0.8705673217773438, 0.8891351222991943, 0.9722214937210083, 0.8038942813873291, 0.7822104692459106, 0.7883211970329285, 0.8124409914016724, 0.8199418783187866, 0.8976427912712097, 0.8753273487091064, 0.8014346361160278, 0.8331345319747925, 0.8581064343452454, 0.8150206804275513, 0.829888641834259, 0.8301893472671509, 0.8170703649520874, 0.7577223181724548, 0.7726888656616211, 0.7968741655349731, 0.8189316391944885, 0.820919394493103], [0.9455858469009399, 1.0, 0.9564516544342041, 0.8817498683929443, 0.8290425539016724, 0.9268245697021484, 0.9358602166175842, 0.8994765877723694, 0.91685551404953, 0.8586713671684265, 0.958881139755249, 0.9730363488197327, 0.9118581414222717, 0.918980062007904, 0.9177203178405762, 0.9352853298187256, 0

In [129]:
from operator import itemgetter
bestmatches=[]
for i,this_sims in enumerate(sims):
    withindex=[(i,sim) for i,sim in enumerate(this_sims)]
    sortedsims=sorted(withindex,key=itemgetter(1),reverse=True)
    mostsim=sortedsims[1][0]
    bestmatches.append((i,mostsim,sortedsims[1][1]))
    
print(bestmatches)

[(0, 15, 0.9722214937210083), (1, 11, 0.9730363488197327), (2, 10, 0.9718309640884399), (3, 13, 0.9677866697311401), (4, 3, 0.9396193623542786), (5, 6, 0.9734714031219482), (6, 5, 0.9734714031219482), (7, 0, 0.9353492856025696), (8, 14, 0.9927843809127808), (9, 21, 0.951431155204773), (10, 11, 0.9748112559318542), (11, 10, 0.9748112559318542), (12, 13, 0.9816993474960327), (13, 12, 0.9816993474960327), (14, 8, 0.9927843809127808), (15, 0, 0.9722214937210083), (16, 17, 0.9780152440071106), (17, 18, 0.9904727339744568), (18, 17, 0.9904727339744568), (19, 20, 0.9882210493087769), (20, 19, 0.9882210493087769), (21, 22, 0.9560396075248718), (22, 21, 0.9560396075248718), (23, 32, 0.978070080280304), (24, 23, 0.9554323554039001), (25, 27, 0.9564154148101807), (26, 30, 0.9380810260772705), (27, 34, 0.995181679725647), (28, 27, 0.9593014717102051), (29, 21, 0.9392608404159546), (30, 32, 0.9517409205436707), (31, 32, 0.9511420130729675), (32, 23, 0.978070080280304), (33, 27, 0.9856246113777161),

In [130]:
# what I actually asked for in Q2!!
bestmatches=[]
list_of_sims=[]
for i,this_sims in enumerate(sims):
    for j,sim in enumerate(this_sims):
        if i<j:
            list_of_sims.append((i,j,sim))

    
sortedsims=sorted(list_of_sims,key=itemgetter(2),reverse=True)
bestmatches=sortedsims[:10]
print(bestmatches)

[(27, 34, 0.995181679725647), (8, 14, 0.9927843809127808), (17, 18, 0.9904727339744568), (19, 20, 0.9882210493087769), (27, 33, 0.9856246113777161), (12, 13, 0.9816993474960327), (33, 34, 0.9796446561813354), (23, 32, 0.978070080280304), (16, 17, 0.9780152440071106), (16, 20, 0.9770832657814026)]


In [131]:
for a,b,c in bestmatches:
    print(f"{inputsents[a]} : {inputsents[b]} ({c})")

[CLS] I can see a boy kicking a ball. [SEP] : [CLS] I can see a girl kicking a ball. [SEP] (0.995181679725647)
[CLS] The male child kicks the ball. [SEP] : [CLS] The female child kicks the ball. [SEP] (0.9927843809127808)
[CLS] The female child plays with dolls. [SEP] : [CLS] The male child plays with dolls. [SEP] (0.9904727339744568)
[CLS] The girl plays with dolls. [SEP] : [CLS] The boy plays with dolls. [SEP] (0.9882210493087769)
[CLS] I can see a boy kicking a ball. [SEP] : [CLS] I can see a male child kicking a ball. [SEP] (0.9856246113777161)
[CLS] The boy is hit by the ball. [SEP] : [CLS] The ball is hit by the boy. [SEP] (0.9816993474960327)
[CLS] I can see a male child kicking a ball. [SEP] : [CLS] I can see a girl kicking a ball. [SEP] (0.9796446561813354)
[CLS] All boys kick balls. [SEP] : [CLS] Boys always kick balls. [SEP] (0.978070080280304)
[CLS] The child plays with dolls. [SEP] : [CLS] The female child plays with dolls. [SEP] (0.9780152440071106)
[CLS] The child plays 

### Exercise 3  

a) Repeat exercise 2 but use the centroid of all of the output embeddings as the representation of a sentence.

b) Experiment with using different pooling layers from the hidden state embeddings.  Typically, using the penultimate layer (-2) is felt to be optimal as it is far enough away from the original uncontextualised word embeddings but also not too close to the output predictions.

In [132]:
def encode(tokenized_sents, method="sum",poolinglayer=-1):
    model = BertModel.from_pretrained('bert-base-uncased')
    model.eval()
    encoded=[]
    blacklist=['[CLS]','[SEP]']
    for sent in tokenized_sents:
        indexed_tokens = tokenizer.convert_tokens_to_ids(sent)
        segment_ids=make_segment_ids(sent)
        tokens_tensor = torch.tensor([indexed_tokens])
        segments_tensors = torch.tensor([segment_ids])

        # Predict all tokens
        with torch.no_grad():
            outputs = model(tokens_tensor, token_type_ids=segments_tensors,output_hidden_states=True)
            if poolinglayer==0:
                predictions = outputs[0]
            else:
                predictions = outputs[2][poolinglayer]
                
        if method=="sum":
            rep=sum(predictions[0])
        elif method=="cls":
            rep=predictions[0][0]
        elif method=="centroid":
            rep=sum(predictions[0])
            rep=rep/len(predictions[0])
        elif method=="centroid-":  #don't include CLS or SEP token representations
            rep=predictions[0][1]
            length=1
            for tok,pred in zip(sent[2:],predictions[0][2:]):
                if tok not in blacklist:
                    rep+=pred
                    length+=1
            rep=rep/length
                    
        
        else:
            rep=predictions[0][0]
        encoded.append(rep)
        
    return encoded

def allpairssims(encoded):
    sims=[]
    for posA in encoded:
        this_sims=[]
        for posB in encoded:
        
            output=cos(posA,posB)
            this_sims.append(output.item())
        sims.append(this_sims)
    return sims

def match(inputsents,sims):
    #best match for each sentence not 10 best matches as asked for in the question!
    bestmatches=[]
    for i,this_sims in enumerate(sims):
        withindex=[(i,sim) for i,sim in enumerate(this_sims)]
        sortedsims=sorted(withindex,key=itemgetter(1),reverse=True)
        mostsim=sortedsims[1][0]
        bestmatches.append((i,mostsim,sortedsims[1][1]))
    
    for a,b,c in bestmatches:
        print(inputsents[a]+" : "+inputsents[b]+":"+str(c))
        
def run(sentences,method="sum",poolinglayer=-1):
    inputsents=['[CLS] '+sent+' [SEP]' for sent in sentences]

    tokenized_sents=[tokenizer.tokenize(sent) for sent in inputsents]
    encoded=encode(tokenized_sents,method=method,poolinglayer=poolinglayer)
    sims=allpairssims(encoded)
    match(sentences,sims)
    return sims
    

In [133]:
sims=run(sentences,poolinglayer=-2)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14208.06it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

The boy kicks the ball. : The boy hits the ball.:0.9567534923553467
The ball kicks the boy. : The ball hits the boy.:0.9486831426620483
The child kicks the ball. : The male child kicks the ball.:0.940710723400116
The ball is kicked by the boy. : The ball is hit by the boy.:0.9381664991378784
The ball is kicked. : The ball is kicked by the boy.:0.8978032469749451
The boy kicks. : The child kicks.:0.9443667531013489
The child kicks. : The boy kicks.:0.9443667531013489
The boy kicks a round object. : The boy kicks the ball.:0.8538012504577637
The male child kicks the ball. : The female child kicks the ball.:0.9862251877784729
The boy is playing football. : The boy is kicking the ball.:0.8970392346382141
The boy hits the ball. : The boy kicks the ball.:0.9567534923553467
The ball hits the boy. : The ball kicks the boy.:0.9486831426620483
The boy is hit by the ball. : The ball is hit by the boy.:0.9677596092224121
The ball is hit by the boy. : The boy is hit by the ball.:0.9677596092224121


In [134]:
sims=run(sentences,method="centroid-",poolinglayer=-2)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16236.75it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

The boy kicks the ball. : The boy hits the ball.:0.961533784866333
The ball kicks the boy. : The ball hits the boy.:0.9485033750534058
The child kicks the ball. : The boy kicks the ball.:0.9454086422920227
The ball is kicked by the boy. : The ball is hit by the boy.:0.9388431310653687
The ball is kicked. : The ball is kicked by the boy.:0.8885976076126099
The boy kicks. : The child kicks.:0.9356702566146851
The child kicks. : The boy kicks.:0.9356702566146851
The boy kicks a round object. : The boy kicks the ball.:0.8423945903778076
The male child kicks the ball. : The female child kicks the ball.:0.9846084117889404
The boy is playing football. : The boy is kicking the ball.:0.8888976573944092
The boy hits the ball. : The boy kicks the ball.:0.961533784866333
The ball hits the boy. : The boy hits the ball.:0.9492038488388062
The boy is hit by the ball. : The ball is hit by the boy.:0.9649645686149597
The ball is hit by the boy. : The boy is hit by the ball.:0.9649645686149597
The femal

In [135]:
print(sims[0])

[1.0000001192092896, 0.9354788064956665, 0.9454086422920227, 0.8232787251472473, 0.7298961877822876, 0.8858634829521179, 0.8415005803108215, 0.8423945903778076, 0.8629533648490906, 0.7868733406066895, 0.961533784866333, 0.914025068283081, 0.8350278735160828, 0.8359919190406799, 0.851132333278656, 0.9486181735992432, 0.6974796652793884, 0.6882858276367188, 0.6899893283843994, 0.7253851294517517, 0.7350742220878601, 0.8822487592697144, 0.8386595249176025, 0.7114238142967224, 0.7613986730575562, 0.7631334662437439, 0.7219316363334656, 0.7569626569747925, 0.7752488851547241, 0.7863667607307434, 0.6756166815757751, 0.6530611515045166, 0.7036939859390259, 0.7283309102058411, 0.7266929149627686]


In [136]:
interested=[2,1,21,7,8,22,3,25]
for i in interested:
    print(sentences[i],sims[0][i])

The child kicks the ball. 0.9454086422920227
The ball kicks the boy. 0.9354788064956665
The boy is kicking the ball. 0.8822487592697144
The boy kicks a round object. 0.8423945903778076
The male child kicks the ball. 0.8629533648490906
The boy is not kicking the ball. 0.8386595249176025
The ball is kicked by the boy. 0.8232787251472473
There is a boy kicking a ball. 0.7631334662437439


In [137]:
sims=run(sentences,poolinglayer=-1)
for i in interested:
    print(sentences[i],sims[0][i])

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14057.78it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

The boy kicks the ball. : The girl kicks the ball.:0.930867612361908
The ball kicks the boy. : The ball hits the boy.:0.929996132850647
The child kicks the ball. : The boy kicks the ball.:0.9198458194732666
The ball is kicked by the boy. : The ball is hit by the boy.:0.9140605926513672
The ball is kicked. : The ball is kicked by the boy.:0.8828246593475342
The boy kicks. : The child kicks.:0.9363823533058167
The child kicks. : The boy kicks.:0.9363823533058167
The boy kicks a round object. : The boy kicks the ball.:0.863271951675415
The male child kicks the ball. : The female child kicks the ball.:0.9780563116073608
The boy is playing football. : The boy is kicking the ball.:0.880127489566803
The boy hits the ball. : The boy kicks the ball.:0.9300813674926758
The ball hits the boy. : The ball kicks the boy.:0.929996132850647
The boy is hit by the ball. : The ball is hit by the boy.:0.9540541172027588
The ball is hit by the boy. : The boy is hit by the ball.:0.9540541172027588
The femal

In [138]:
sims=run(sentences,poolinglayer=0)
for i in interested:
    print(sentences[i],sims[0][i])

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13452.38it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

The boy kicks the ball. : The girl kicks the ball.:0.930867612361908
The ball kicks the boy. : The ball hits the boy.:0.929996132850647
The child kicks the ball. : The boy kicks the ball.:0.9198458194732666
The ball is kicked by the boy. : The ball is hit by the boy.:0.9140605926513672
The ball is kicked. : The ball is kicked by the boy.:0.8828246593475342
The boy kicks. : The child kicks.:0.9363823533058167
The child kicks. : The boy kicks.:0.9363823533058167
The boy kicks a round object. : The boy kicks the ball.:0.863271951675415
The male child kicks the ball. : The female child kicks the ball.:0.9780563116073608
The boy is playing football. : The boy is kicking the ball.:0.880127489566803
The boy hits the ball. : The boy kicks the ball.:0.9300813674926758
The ball hits the boy. : The ball kicks the boy.:0.929996132850647
The boy is hit by the ball. : The ball is hit by the boy.:0.9540541172027588
The ball is hit by the boy. : The boy is hit by the ball.:0.9540541172027588
The femal

In [139]:
sims=run(sentences,method="cls",poolinglayer=0)
for i in interested:
    print(sentences[i],sims[0][i])

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12563.09it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

The boy kicks the ball. : The girl kicks the ball.:0.9722214937210083
The ball kicks the boy. : The ball hits the boy.:0.9730363488197327
The child kicks the ball. : The boy hits the ball.:0.9718309640884399
The ball is kicked by the boy. : The ball is hit by the boy.:0.9677866697311401
The ball is kicked. : The ball is kicked by the boy.:0.9396193623542786
The boy kicks. : The child kicks.:0.9734714031219482
The child kicks. : The boy kicks.:0.9734714031219482
The boy kicks a round object. : The boy kicks the ball.:0.9353492856025696
The male child kicks the ball. : The female child kicks the ball.:0.9927843809127808
The boy is playing football. : The boy is kicking the ball.:0.951431155204773
The boy hits the ball. : The ball hits the boy.:0.9748112559318542
The ball hits the boy. : The boy hits the ball.:0.9748112559318542
The boy is hit by the ball. : The ball is hit by the boy.:0.9816993474960327
The ball is hit by the boy. : The boy is hit by the ball.:0.9816993474960327
The fema

### Extension 1
The MRPC.zip file contains a training, dev and test split for the Microsoft Research paraphrase corpus.  In this corpus the quality '1' indicates that the 2 sentences are considered to be paraphrases and '0' indicates that they are not.

Can you build a classifier on top of the BERT pre-trained model, trained on the training split of MRPC, which predicts whether 2 sentences are paraphrases or not?

Note this does not require you to fine-tune the BERT model.  You can use outputs from BERT as input to your separate classifier.  I would suggest a single neural layer which uses the representation from exercise 2 or 3 as input, built using scikit-learn or torch.   